# LSST-Only ANTARES Historical Backfill

This notebook builds and extends the cumulative LSST-only ANTARES history store on Rubin Science Platform.

Safety notes:

- Removing old notebook cells does **not** remove saved parquet or manifest data.
- This notebook only writes data when you run the extraction cell in Section 6.
- Saved data live outside GitHub under `/home/ivezic/AntaresAlerts/ANTARES_Analysis_Data`.
- Do not use `/project` in the current RSP session; it may be read-only or unavailable.
- `alerts_time_comparison.ipynb` is the real-time comparison notebook. This notebook is only for historical backfill.


## 1. Imports and RSP Path Setup

Run this first. It keeps the project import path explicit and uses the writable RSP data root.


In [3]:
from pathlib import Path
import hashlib
import json
import math
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from astropy.time import Time
from antares_client.search import search as antares_search, get_by_id

# Make imports robust whether Jupyter starts in the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# RSP storage for this project. Keep heavy parquet data out of GitHub.
os.environ["ANTARES_DATA_ROOT"] = "/home/ivezic/AntaresAlerts/ANTARES_Analysis_Data"
DATA_ROOT = Path(os.environ["ANTARES_DATA_ROOT"])
DATA_ROOT.mkdir(parents=True, exist_ok=True)

from src import history, query

pd.set_option("display.max_columns", 80)

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Survey data root:", history.survey_data_root(DATA_ROOT))


Project root: /home/mdarim/notebooks/ANTARES_Analysis
Data root: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data
Survey data root: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/lsst_only


## 2. User Settings

Change this cell for the nights you want to backfill. MJD windows are half-open: `[MJD_START, MJD_STOP)`. For example, `61096.0 -> 61097.0` is UTC date `2026-02-25`.


In [109]:
# Change these two values for the historical range you want to backfill.
# The notebook processes one full MJD day at a time.
MJD_START = 61176.0
MJD_STOP = 61177.0

# Keep True for normal use. Existing complete nightly partitions are loaded instead of re-queried.
RESUME_EXISTING_NIGHT = True

# Full science run: fetch per-locus ANTARES lightcurves after locus extraction.
FETCH_LIGHTCURVES = True
LIGHTCURVE_WORKERS = 4

# Probe-first extraction. If a tile returns PROBE_THRESHOLD rows, it may contain more data and is split.
PROBE_LIMIT = 50
PROBE_THRESHOLD = 50
TIME_BIN_MINUTES = 30
RA_BINS = 24
DEC_BINS = 6

# Final safety floors. If a tile is still saturated below these, the night fails rather than being accepted incomplete.
MIN_TIME_SECONDS = 30.0
MIN_RA_DEGREES = 0.05
MIN_DEC_DEGREES = 0.05
ALLOW_SATURATED_FINAL_TILES = False

# ANTARES retry/cache behavior.
MAX_QUERY_RETRIES = 6
RETRY_SLEEP_SECONDS = 20
CACHE_VERSION = "probe50_v1"

print("Backfill range:", (MJD_START, MJD_STOP))
print("Date range:", history.mjd_to_utc_date(MJD_START), "to", history.mjd_to_utc_date(MJD_STOP))
print("Fetch lightcurves:", FETCH_LIGHTCURVES)


Backfill range: (61176.0, 61177.0)
Date range: 2026-05-16 to 2026-05-17
Fetch lightcurves: True


## 3. Stored Data Preflight

This is safe to rerun. It rebuilds compact cumulative indexes from saved nightly parquet files and reports what is already stored.


In [110]:
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

SUMMARY_COLUMNS = [
    "date_utc", "display_date", "mjd_min", "mjd_max",
    "actual_loci", "alert_rows", "status", "append_ready",
    "lsst_only_pass", "lsst_dia_count", "lsst_ss_count", "ztf_object_id_count",
]

print(f"Cumulative loci index rows: {len(loci_index):,}")
if nightly_summary.empty:
    print("No nightly partitions are indexed yet.")
else:
    display(nightly_summary[SUMMARY_COLUMNS].sort_values("mjd_min"))

unexpected = nightly_summary[
    ~nightly_summary["date_utc"].astype(str).str.match(r"2026-(02|03|04|05|06|07|08|09|10|11|12)-")
]
if len(unexpected):
    print("WARNING: unexpected date rows are present in the data store. This notebook will not remove them.")
    display(unexpected[SUMMARY_COLUMNS])


Cumulative loci index rows: 373,707


,date_utc,display_date,mjd_min,mjd_max,actual_loci,alert_rows,status,append_ready,lsst_only_pass,lsst_dia_count,lsst_ss_count,ztf_object_id_count
0,2026-02-25,2026/2/25,61096.0,61097.0,113459,1012223,complete,True,True,113459,113459,113459
1,2026-02-26,2026/2/26,61097.0,61098.0,96238,933004,complete,True,True,96238,96238,96238
2,2026-02-27,2026/2/27,61098.0,61099.0,47909,1590484,complete,True,True,47909,47909,47909
3,2026-02-28,2026/2/28,61099.0,61100.0,2919,139630,complete,True,True,2919,2919,2919
4,2026-03-01,2026/3/1,61100.0,61101.0,1361,138097,complete,True,True,1361,1361,1361
...,...,...,...,...,...,...,...,...,...,...,...,...
63,2026-05-25,2026/5/25,61185.0,61186.0,49926,1527986,complete,True,True,49926,49926,49926
64,2026-05-27,2026/5/27,61187.0,61188.0,25092,254055,complete,True,True,25092,25092,25092
65,2026-05-30,2026/5/30,61190.0,61191.0,2921,754340,complete,True,True,2077,862,288
66,2026-06-02,2026/6/2,61193.0,61194.0,623,511963,complete,True,True,623,1,623


## 4. Probe-First Backfill Helpers

These helpers avoid deep ANTARES pagination. A tile is accepted only when it returns fewer than `PROBE_THRESHOLD` loci. Dense tiles split in time, RA, or Dec until they are safe or hit the final safety floors.


In [111]:
MJD_COL = "newest_alert_observation_time"
LOCUS_ID_COL = "locus_id"


def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def tile_cache_path(cache_root, tile):
    raw = json.dumps(tile, sort_keys=True).encode("utf-8")
    key = hashlib.sha1(raw).hexdigest()
    return cache_root / f"tile_{key}.parquet"


def lightcurve_cache_path(cache_root, locus_id):
    key = hashlib.sha1(str(locus_id).encode("utf-8")).hexdigest()
    return cache_root / f"lc_{key}.parquet"


def build_tile_query(tile):
    dec_upper_op = "lte" if float(tile["dec_max"]) >= 90.0 else "lt"
    return {
        "query": {
            "bool": {
                "filter": [
                    {
                        "range": {
                            "properties.newest_alert_observation_time": {
                                "gte": float(tile["mjd_min"]),
                                "lt": float(tile["mjd_max"]),
                            }
                        }
                    },
                    {"range": {"ra": {"gte": float(tile["ra_min"]), "lt": float(tile["ra_max"])}}},
                    {"range": {"dec": {"gte": float(tile["dec_min"]), dec_upper_op: float(tile["dec_max"])}}},
                    query.lsst_identifier_filter(),
                ]
            }
        }
    }


def collect_tile(tile, limit):
    records = []
    for locus in antares_search(build_tile_query(tile)):
        records.append(query.locus_to_record(locus))
        if len(records) >= int(limit):
            break
    return pd.DataFrame(records)


def collect_tile_with_retries(tile, limit, max_retries=MAX_QUERY_RETRIES):
    for attempt in range(1, max_retries + 1):
        try:
            return collect_tile(tile, limit)
        except Exception as exc:
            if attempt == max_retries:
                raise
            wait = RETRY_SLEEP_SECONDS * attempt
            print(f"    query failed attempt {attempt}/{max_retries}: {exc}")
            print(f"    sleeping {wait}s")
            time.sleep(wait)


def make_initial_tiles(mjd_start, mjd_stop):
    step = TIME_BIN_MINUTES / 1440.0
    mjd_edges = list(np.arange(float(mjd_start), float(mjd_stop), step)) + [float(mjd_stop)]
    mjd_edges = sorted(set(round(x, 12) for x in mjd_edges))
    if mjd_edges[-1] < float(mjd_stop):
        mjd_edges.append(float(mjd_stop))

    ra_edges = np.linspace(0.0, 360.0, RA_BINS + 1)
    dec_edges = np.linspace(-90.0, 90.0, DEC_BINS + 1)

    tiles = []
    for lo, hi in zip(mjd_edges[:-1], mjd_edges[1:]):
        if hi <= lo:
            continue
        for ra0, ra1 in zip(ra_edges[:-1], ra_edges[1:]):
            for dec0, dec1 in zip(dec_edges[:-1], dec_edges[1:]):
                tiles.append({
                    "mjd_min": float(lo), "mjd_max": float(hi),
                    "ra_min": float(ra0), "ra_max": float(ra1),
                    "dec_min": float(dec0), "dec_max": float(dec1),
                })
    return tiles


def split_tile(tile):
    time_seconds = (float(tile["mjd_max"]) - float(tile["mjd_min"])) * 86400.0
    ra_width = float(tile["ra_max"]) - float(tile["ra_min"])
    dec_width = float(tile["dec_max"]) - float(tile["dec_min"])

    ratios = {
        "time": time_seconds / MIN_TIME_SECONDS,
        "ra": ra_width / MIN_RA_DEGREES,
        "dec": dec_width / MIN_DEC_DEGREES,
    }
    dim = max(ratios, key=ratios.get)
    if ratios[dim] <= 1.0:
        return []

    first = dict(tile)
    second = dict(tile)
    if dim == "time":
        mid = (float(tile["mjd_min"]) + float(tile["mjd_max"])) / 2.0
        first["mjd_max"] = mid
        second["mjd_min"] = mid
    elif dim == "ra":
        mid = (float(tile["ra_min"]) + float(tile["ra_max"])) / 2.0
        first["ra_max"] = mid
        second["ra_min"] = mid
    else:
        mid = (float(tile["dec_min"]) + float(tile["dec_max"])) / 2.0
        first["dec_max"] = mid
        second["dec_min"] = mid
    return [first, second]


def extract_full_loci_probe_first(mjd_start, mjd_stop, date_utc):
    cache_root = DATA_ROOT / "cache" / CACHE_VERSION / "tile_loci" / date_utc
    cache_root.mkdir(parents=True, exist_ok=True)

    pending = make_initial_tiles(mjd_start, mjd_stop)
    accepted = []
    saturated = []
    report = []
    done = 0
    t0 = time.time()

    print("Stage 1: probe-first LSST-only locus extraction")
    print(f"Initial tiles: {len(pending):,}")
    print(f"Probe limit: {PROBE_LIMIT}; split if rows >= {PROBE_THRESHOLD}")

    while pending:
        tile = pending.pop(0)
        done += 1
        cpath = tile_cache_path(cache_root, tile)

        if cpath.exists():
            df = pd.read_parquet(cpath)
            source = "cache"
        else:
            df = collect_tile_with_retries(tile, PROBE_LIMIT)
            df.to_parquet(cpath, index=False)
            source = "live"

        n_rows = len(df)
        action = "accepted"
        if n_rows >= PROBE_THRESHOLD:
            children = split_tile(tile)
            if children:
                pending = children + pending
                action = "split"
            else:
                saturated.append(tile)
                accepted.append(df)
                action = "saturated_final"
        else:
            accepted.append(df)

        report.append({**tile, "rows": int(n_rows), "action": action, "source": source})

        should_print = done <= 20 or done % 250 == 0 or action != "accepted" or n_rows > 0 or not pending
        if should_print:
            elapsed = max(time.time() - t0, 1e-6)
            rate = done / elapsed
            print(
                f"{done:7d} done | {len(pending):7d} queued | {rate:4.1f} tiles/s | "
                f"MJD {tile['mjd_min']:.6f}-{tile['mjd_max']:.6f} "
                f"RA {tile['ra_min']:6.1f}-{tile['ra_max']:6.1f} "
                f"Dec {tile['dec_min']:6.1f}-{tile['dec_max']:6.1f} "
                f"{n_rows:4d} loci {action} ({source})"
            )

    if saturated and not ALLOW_SATURATED_FINAL_TILES:
        raise RuntimeError(
            f"{date_utc} still has {len(saturated)} saturated final tiles. "
            "Lower MIN_TIME_SECONDS, MIN_RA_DEGREES, or MIN_DEC_DEGREES before accepting this night."
        )

    loci = pd.concat(accepted, ignore_index=True, sort=False) if accepted else pd.DataFrame()
    if LOCUS_ID_COL in loci.columns:
        loci = loci.drop_duplicates(subset=[LOCUS_ID_COL], keep="last").reset_index(drop=True)

    print(f"Loci complete: {len(loci):,}")
    return loci, pd.DataFrame(report)


def fetch_one_lightcurve_cached(locus_id, label, cache_root):
    cpath = lightcurve_cache_path(cache_root, locus_id)
    if cpath.exists():
        df = pd.read_parquet(cpath)
        return None if df.empty else df

    locus = get_by_id(locus_id)
    lc = locus.lightcurve
    if lc is None or lc.empty:
        df = pd.DataFrame()
    else:
        df = lc.copy()
        df[LOCUS_ID_COL] = locus_id
        df["range_label"] = label

    df.to_parquet(cpath, index=False)
    return None if df.empty else df


def fetch_lightcurves_cached(df_loci, date_utc, label):
    if df_loci.empty or LOCUS_ID_COL not in df_loci.columns:
        return pd.DataFrame()

    ids = df_loci[LOCUS_ID_COL].dropna().astype(str).drop_duplicates().tolist()
    cache_root = DATA_ROOT / "cache" / CACHE_VERSION / "lightcurves" / date_utc
    cache_root.mkdir(parents=True, exist_ok=True)

    print("Stage 2: full lightcurve extraction")
    print(f"Fetching {len(ids):,} lightcurves with {LIGHTCURVE_WORKERS} workers")

    frames = []
    failures = 0
    done = 0
    with ThreadPoolExecutor(max_workers=LIGHTCURVE_WORKERS) as pool:
        futures = {
            pool.submit(fetch_one_lightcurve_cached, locus_id, label, cache_root): locus_id
            for locus_id in ids
        }
        for future in as_completed(futures):
            done += 1
            try:
                result = future.result()
                if result is not None and not result.empty:
                    frames.append(result)
            except Exception as exc:
                failures += 1
                print(f"    lightcurve failed for {futures[future]}: {exc}")
            if done % 100 == 0 or done == len(ids):
                print(f"{done}/{len(ids)} lightcurves complete; failures={failures}")

    if failures:
        raise RuntimeError(f"Lightcurve extraction had {failures} failures. Rerun with cache/resume before accepting.")

    alerts = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    print(f"Lightcurves complete: {len(alerts):,} alert/lightcurve rows")
    return alerts


## 5. Nightly Ingestion Function

This function writes one nightly partition only after validation passes. Existing complete partitions are loaded when `RESUME_EXISTING_NIGHT=True`.


In [112]:
def manifest_is_complete(manifest, paths):
    if not manifest:
        return False
    return (
        manifest.get("status") == "complete"
        and manifest.get("validation", {}).get("append_ready") is True
        and paths["loci"].exists()
        and paths["alerts"].exists()
    )


def ingest_full_history_night(mjd_min, mjd_max):
    date_utc = history.mjd_to_utc_date(mjd_min)
    label = f"History {history.display_date(date_utc)}"
    paths = history.nightly_paths(DATA_ROOT, date_utc)

    if RESUME_EXISTING_NIGHT and paths["manifest"].exists():
        manifest = history.read_manifest(DATA_ROOT, date_utc)
        if manifest_is_complete(manifest, paths):
            print(f"Resume: loading complete stored partition for {date_utc}")
            df_loci = pd.read_parquet(paths["loci"])
            df_alerts = pd.read_parquet(paths["alerts"])
            return manifest, df_loci, df_alerts, pd.DataFrame(), True
        raise RuntimeError(
            f"Existing partition for {date_utc} is not complete/append-ready. "
            "Inspect or repair it before rerunning this notebook."
        )

    started_at = now_utc()
    t0 = time.time()

    raw_loci, report_df = extract_full_loci_probe_first(mjd_min, mjd_max, date_utc)
    ingested_at = now_utc()
    df_loci = history.prepare_loci(raw_loci, date_utc, mjd_min, mjd_max, ingested_at)
    df_loci["source_query_mode"] = "probe_first_time_ra_dec"

    if FETCH_LIGHTCURVES:
        raw_alerts = fetch_lightcurves_cached(df_loci, date_utc, label)
    else:
        print("Lightcurve fetch disabled for this run.")
        raw_alerts = pd.DataFrame()

    df_alerts = history.prepare_alerts(raw_alerts, date_utc, label)
    validation = history.validation_summary(
        df_loci,
        df_alerts,
        mjd_min=mjd_min,
        mjd_max=mjd_max,
        lsst_only=True,
    )
    if not validation.get("append_ready"):
        raise RuntimeError(f"Validation failed for {date_utc}; no nightly files were written: {validation}")

    paths["dir"].mkdir(parents=True, exist_ok=True)
    df_loci.to_parquet(paths["loci"], index=False)
    df_alerts.to_parquet(paths["alerts"], index=False)

    action_counts = report_df["action"].value_counts().to_dict() if not report_df.empty else {}
    survey_counts = query.lsst_identifier_counts(df_loci)
    manifest = {
        "date_utc": date_utc,
        "mjd_min": float(mjd_min),
        "mjd_max": float(mjd_max),
        "query_tag": None,
        "target_loci": None,
        "actual_loci": int(len(df_loci)),
        "alert_rows": int(len(df_alerts)),
        "chunk_count": int(action_counts.get("accepted", 0)),
        "split_count": int(action_counts.get("split", 0)),
        "saturated_chunk_count": int(action_counts.get("saturated_final", 0)),
        "status": "complete",
        "survey_mode": "lsst",
        "lsst_filter_used": True,
        "lsst_filter": query.lsst_identifier_filter(),
        "parallel_shards": 1,
        "lsst_dia_count": survey_counts["lsst_dia_count"],
        "lsst_ss_count": survey_counts["lsst_ss_count"],
        "ztf_object_id_count": survey_counts["ztf_object_id_count"],
        "started_at_utc": started_at,
        "finished_at_utc": now_utc(),
        "runtime_seconds": round(time.time() - t0, 2),
        "validation": validation,
        "paths": {
            "loci": str(paths["loci"]),
            "alerts": str(paths["alerts"]),
            "manifest": str(paths["manifest"]),
        },
        "extraction_method": {
            "name": "probe_first_time_ra_dec",
            "probe_limit": PROBE_LIMIT,
            "probe_threshold": PROBE_THRESHOLD,
            "time_bin_minutes": TIME_BIN_MINUTES,
            "ra_bins": RA_BINS,
            "dec_bins": DEC_BINS,
            "min_time_seconds": MIN_TIME_SECONDS,
            "min_ra_degrees": MIN_RA_DEGREES,
            "min_dec_degrees": MIN_DEC_DEGREES,
            "cache_version": CACHE_VERSION,
        },
    }

    with open(paths["manifest"], "w") as handle:
        json.dump(manifest, handle, indent=2, sort_keys=True)
        handle.write("\n")

    return manifest, df_loci, df_alerts, report_df, False


## 6. Run Backfill Range

This is the main execution cell. It processes one night at a time and stops on the first failure so incomplete data are not silently accepted.


In [116]:
completed = []
failed = []
last_manifest = None
last_loci = pd.DataFrame()
last_alerts = pd.DataFrame()
last_report = pd.DataFrame()

for date_utc, lo, hi in history.iter_night_windows(MJD_START, MJD_STOP):
    print("\n" + "=" * 80)
    print(f"Backfilling {date_utc}  MJD [{lo:.6f}, {hi:.6f})")

    try:
        manifest, df_loci, df_alerts, report_df, resumed = ingest_full_history_night(lo, hi)
        last_manifest = manifest
        last_loci = df_loci
        last_alerts = df_alerts
        last_report = report_df
        completed.append({
            "date_utc": date_utc,
            "mjd_min": lo,
            "mjd_max": hi,
            "resumed": resumed,
            "loci": len(df_loci),
            "alert_rows": len(df_alerts),
            "status": manifest.get("status"),
        })
        history.update_cumulative_indexes(DATA_ROOT)
        print(f"COMPLETE {date_utc}: loci={len(df_loci):,}, alert_rows={len(df_alerts):,}, resumed={resumed}")
    except Exception as exc:
        failed.append({"date_utc": date_utc, "mjd_min": lo, "mjd_max": hi, "error": str(exc)})
        print(f"FAILED {date_utc}: {exc}")
        break

loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("\n" + "=" * 80)
print("BACKFILL RANGE FINISHED")
print("Completed nights:", len(completed))
print("Failed nights:", len(failed))
print("Cumulative loci rows:", f"{len(loci_index):,}")

display(pd.DataFrame(completed))
if failed:
    display(pd.DataFrame(failed))



Backfilling 2026-05-16  MJD [61176.000000, 61177.000000)
Stage 1: probe-first LSST-only locus extraction
Initial tiles: 6,912
Probe limit: 50; split if rows >= 50
      1 done |    6911 queued | 212.9 tiles/s | MJD 61176.000000-61176.020833 RA    0.0-  15.0 Dec  -90.0- -60.0    0 loci accepted (cache)
      2 done |    6910 queued | 208.5 tiles/s | MJD 61176.000000-61176.020833 RA    0.0-  15.0 Dec  -60.0- -30.0    0 loci accepted (cache)
      3 done |    6909 queued | 204.5 tiles/s | MJD 61176.000000-61176.020833 RA    0.0-  15.0 Dec  -30.0-   0.0    0 loci accepted (cache)
      4 done |    6908 queued | 203.0 tiles/s | MJD 61176.000000-61176.020833 RA    0.0-  15.0 Dec    0.0-  30.0    0 loci accepted (cache)
      5 done |    6907 queued | 207.4 tiles/s | MJD 61176.000000-61176.020833 RA    0.0-  15.0 Dec   30.0-  60.0    0 loci accepted (cache)
      6 done |    6906 queued | 210.2 tiles/s | MJD 61176.000000-61176.020833 RA    0.0-  15.0 Dec   60.0-  90.0    0 loci accepted (cac

,date_utc,mjd_min,mjd_max,resumed,loci,alert_rows,status
0,2026-05-16,61176.0,61177.0,False,665,9025,complete


## 7. Post-Run Diagnostics

Safe to rerun. This section reads saved files and indexes; it does not query ANTARES or remove data.


In [117]:
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print(f"Cumulative loci rows: {len(loci_index):,}")
display(nightly_summary[SUMMARY_COLUMNS].sort_values("mjd_min").tail(30))

if last_manifest is None:
    print("No night was run in this kernel yet. Diagnostics below use the latest indexed night if available.")
    if nightly_summary.empty:
        raise RuntimeError("No nightly summary rows are available.")
    latest_row = nightly_summary.sort_values("mjd_min").iloc[-1]
    diagnostic_date = latest_row["date_utc"]
    diagnostic_manifest = history.read_manifest(DATA_ROOT, diagnostic_date)
else:
    diagnostic_date = last_manifest["date_utc"]
    diagnostic_manifest = last_manifest

paths = history.nightly_paths(DATA_ROOT, diagnostic_date)
print("\nDiagnostic night:", diagnostic_date)
print("Manifest exists:", paths["manifest"].exists(), paths["manifest"])
print("Loci parquet exists:", paths["loci"].exists(), paths["loci"])
print("Alerts parquet exists:", paths["alerts"].exists(), paths["alerts"])

if not (paths["manifest"].exists() and paths["loci"].exists() and paths["alerts"].exists()):
    raise RuntimeError("One or more expected nightly files are missing.")

check_loci = pd.read_parquet(paths["loci"])
check_alerts = pd.read_parquet(paths["alerts"])

print("\nManifest status:", diagnostic_manifest.get("status"))
print("Manifest append_ready:", diagnostic_manifest.get("validation", {}).get("append_ready"))
print("Manifest LSST-only pass:", diagnostic_manifest.get("validation", {}).get("lsst_only_pass"))
print("Manifest loci:", diagnostic_manifest.get("actual_loci"), "| Parquet loci:", len(check_loci))
print("Manifest alert rows:", diagnostic_manifest.get("alert_rows"), "| Parquet alert rows:", len(check_alerts))

if int(diagnostic_manifest.get("actual_loci", -1)) != len(check_loci):
    raise RuntimeError("Manifest loci count does not match loci parquet row count.")
if int(diagnostic_manifest.get("alert_rows", -1)) != len(check_alerts):
    raise RuntimeError("Manifest alert count does not match alerts parquet row count.")

survey_counts = query.lsst_identifier_counts(check_loci)
print("\nLSST/ZTF identifier counts:")
for key, value in survey_counts.items():
    print(f"  {key}: {value:,}")

print("\nDiagnostics passed.")


Cumulative loci rows: 374,372


,date_utc,display_date,mjd_min,mjd_max,actual_loci,alert_rows,status,append_ready,lsst_only_pass,lsst_dia_count,lsst_ss_count,ztf_object_id_count
39,2026-04-10,2026/4/10,61140.0,61141.0,1222,7400,complete,True,True,412,826,11
40,2026-04-12,2026/4/12,61142.0,61143.0,830,11187,complete,True,True,287,551,11
41,2026-04-14,2026/4/14,61144.0,61145.0,852,2435,complete,True,True,852,852,852
42,2026-04-15,2026/4/15,61145.0,61146.0,1131,11531,complete,True,True,313,829,32
43,2026-04-16,2026/4/16,61146.0,61147.0,1266,11505,complete,True,True,1266,1266,1266
44,2026-04-17,2026/4/17,61147.0,61148.0,9,2459,complete,True,True,9,0,9
45,2026-04-18,2026/4/18,61148.0,61149.0,7,4217,complete,True,True,6,1,7
46,2026-04-20,2026/4/20,61150.0,61151.0,10,1880,complete,True,True,10,0,10
47,2026-04-21,2026/4/21,61151.0,61152.0,1612,45837,complete,True,True,1612,1612,1612
48,2026-04-22,2026/4/22,61152.0,61153.0,660,4439,complete,True,True,171,499,1



Diagnostic night: 2026-05-16
Manifest exists: True /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/05/16/manifest.json
Loci parquet exists: True /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/05/16/loci.parquet
Alerts parquet exists: True /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/lsst_only/nightly/2026/05/16/alerts.parquet

Manifest status: complete
Manifest append_ready: True
Manifest LSST-only pass: True
Manifest loci: 665 | Parquet loci: 665
Manifest alert rows: 9025 | Parquet alert rows: 9025

LSST/ZTF identifier counts:
  lsst_dia_count: 665
  lsst_ss_count: 665
  lsst_identifier_count: 665
  ztf_object_id_count: 665

Diagnostics passed.


In [118]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/home/mdarim/notebooks/ANTARES_Analysis")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import history

DATA_ROOT = Path("/home/ivezic/AntaresAlerts/ANTARES_Analysis_Data")
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)

print("Cumulative loci rows:", len(loci_index))
display(nightly_summary.tail(10))

Cumulative loci rows: 374372


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,saturated_chunk_count,status,append_ready,mjd_pass,duplicate_locus_count,coordinate_pass,overlap_count,alert_locus_link_pass,survey_mode,lsst_filter_used,parallel_shards,lsst_dia_count,lsst_ss_count,ztf_object_id_count,lsst_only_pass,history_start_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
59,2026-05-11,2026/5/11,61171.0,61172.0,None,None,2,702,6912,0,0,complete,True,True,0,True,0,True,lsst,True,1,2,0,2,True,True,1691.99,2026-06-10T23:15:58+00:00,2026-06-10T23:44:10+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
60,2026-05-12,2026/5/12,61172.0,61173.0,None,None,3220,48316,7054,142,0,complete,True,True,0,True,0,True,lsst,True,1,3220,3220,3220,True,True,33.69,2026-06-11T01:33:40+00:00,2026-06-11T01:34:14+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
61,2026-05-14,2026/5/14,61174.0,61175.0,None,None,1,86,6912,0,0,complete,True,True,0,True,0,True,lsst,True,1,1,0,1,True,True,910.52,2026-06-11T02:20:25+00:00,2026-06-11T02:35:36+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
62,2026-05-15,2026/5/15,61175.0,61176.0,None,None,10,4786,6912,0,0,complete,True,True,0,True,0,True,lsst,True,1,10,0,10,True,True,925.84,2026-06-11T02:50:18+00:00,2026-06-11T03:05:44+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
63,2026-05-16,2026/5/16,61176.0,61177.0,None,None,665,9025,6952,40,0,complete,True,True,0,True,0,True,lsst,True,1,665,665,665,True,True,32.02,2026-06-11T04:05:24+00:00,2026-06-11T04:05:56+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
64,2026-05-25,2026/5/25,61185.0,61186.0,None,None,49926,1527986,9625,2713,0,complete,True,True,0,True,0,True,lsst,True,1,49926,49926,49926,True,True,199.40,2026-05-26T18:25:29+00:00,2026-05-26T18:28:48+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
65,2026-05-27,2026/5/27,61187.0,61188.0,None,None,25092,254055,8279,1367,0,complete,True,True,0,True,0,True,lsst,True,1,25092,25092,25092,True,True,123.16,2026-05-28T15:11:27+00:00,2026-05-28T15:13:30+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
66,2026-05-30,2026/5/30,61190.0,61191.0,None,None,2921,754340,7048,136,0,complete,True,True,0,True,0,True,lsst,True,1,2077,862,288,True,True,1547.69,2026-05-31T08:39:39+00:00,2026-05-31T09:05:27+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
67,2026-06-02,2026/6/2,61193.0,61194.0,None,None,623,511963,6928,16,0,complete,True,True,0,True,0,True,lsst,True,1,623,1,623,True,True,1175.18,2026-06-03T03:52:55+00:00,2026-06-03T04:12:30+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
68,2026-06-03,2026/6/3,61194.0,61195.0,None,None,876,704237,6937,25,0,complete,True,True,0,True,0,True,lsst,True,1,876,0,876,True,True,1366.07,2026-06-04T02:35:38+00:00,2026-06-04T02:58:24+00:00,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...,/home/ivezic/AntaresAlerts/ANTARES_Analysis_Da...
